# Do weight edits cost knowledge?

Reproduces the retention test in this repository on a free Colab T4 GPU (about 26 minutes).

Before running: **Runtime > Change runtime type > T4 GPU**.

Cell 1 asks for a zip of this repository. On the GitHub page choose **Code > Download ZIP** and upload that file.

In [ ]:
import torch, os, glob, zipfile
assert torch.cuda.is_available(), 'No GPU. Use Runtime > Change runtime type > T4 GPU, then run again.'
print(torch.cuda.get_device_name(0))
from google.colab import files
hits = glob.glob('/content/**/run_retention.py', recursive=True)
if not hits:
    up = files.upload()   # choose the repository zip
    for name in up:
        zipfile.ZipFile(name).extractall('/content/repo')
    hits = glob.glob('/content/**/run_retention.py', recursive=True)
REPO = os.path.dirname(hits[0])
print('repository at', REPO)

In [ ]:
!pip install -q -U transformers datasets safetensors

## Smoke test: 50 questions

Every `mit_` line must say `sha256 OK`, every `ws_` line must say `matches paper`, and the example prompt must end with `The answer is (`.

In [ ]:
os.chdir(REPO)
!python run_retention.py --limit 50 --out results_smoke
!python analyze.py results_smoke

## Full run: 1,000 questions, base + 10 variants

Writes to `results_rerun/` so the published result files in `results/` stay untouched.

In [ ]:
!python run_retention.py --out results_rerun

In [ ]:
!python analyze.py results_rerun | tee results_rerun/analysis.txt
!python extra_checks.py results_rerun

## Compare with the published results

The scoring is deterministic, so predictions should match the published files (tiny differences are possible across GPU types).

In [ ]:
import json
for f in sorted(glob.glob('results/*.jsonl')):
    a = [json.loads(l)['pred'] for l in open(f)]
    b = [json.loads(l)['pred'] for l in open(f.replace('results/', 'results_rerun/'))]
    print(os.path.basename(f), 'same predictions:', sum(x == y for x, y in zip(a, b)), '/', len(a))